In [1]:
import sys
from pathlib import Path

project_root = Path().resolve().parent

# путь до src
src_path = project_root / "src"

# добавляем src в пути импорта
if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

print("PROJECT ROOT:", project_root)
print("SRC PATH:", src_path)

PROJECT ROOT: C:\Users\denis\credit_scoring
SRC PATH: C:\Users\denis\credit_scoring\src


In [2]:
from config.db_config import DB_ARGS, DATA_FULL_PATH
from app.utils.db_manager import PostgresDB

db = PostgresDB(DB_ARGS)
db.connect()

Подключение к БД успешно установлено


In [3]:
import os
from datetime import datetime

import pandas as pd
import psycopg2

In [4]:
from warnings import filterwarnings

filterwarnings('ignore', category=UserWarning, message='.*pandas only supports SQLAlchemy connectable.*')

In [6]:
# вывести средний доход среди всех клиентов
db.get_df("""
SELECT ROUND(AVG(AMT_INCOME_TOTAL)::numeric, 2) AS avg_income
FROM application;
""")

Время выполнения: 0:00:00


,avg_income
0,170116.06


In [ ]:
# вывести минимальный и максимальный возраст среди всех клиентов
db.get_df("""
SELECT
    FLOOR(MIN(ABS(DAYS_BIRTH)) / 365.25) AS min_age,
    FLOOR(MAX(ABS(DAYS_BIRTH)) / 365.25) AS max_age
FROM application;
""")

Время выполнения: 0:00:00


,min_age,max_age
0,20.0,69.0


In [ ]:
# вывести количество мужчин и женщин
db.get_df("""
SELECT
    CODE_GENDER,
    COUNT(*) AS client_count
FROM application
WHERE CODE_GENDER IN ('M', 'F')
GROUP BY CODE_GENDER
ORDER BY CODE_GENDER;
""")

Время выполнения: 0:00:00


,code_gender,client_count
0,F,235126
1,M,121125


In [ ]:
# вывести общую сумму, количество и среднюю сумму, запрошенную клиентами в кредит с авто и без
db.get_df("""
SELECT
    FLAG_OWN_CAR,
    COUNT(*) AS client_count,
    ROUND(SUM(AMT_CREDIT)::numeric, 2) AS total_credit_amount,
    ROUND(AVG(AMT_CREDIT)::numeric, 2) AS avg_credit_amount
FROM application
GROUP BY FLAG_OWN_CAR
ORDER BY FLAG_OWN_CAR;
""")

Время выполнения: 0:00:00


,flag_own_car,client_count,total_credit_amount,avg_credit_amount
0,N,235235,1.304050e+11,554317.38
1,Y,121020,7.900260e+10,652786.57


In [ ]:
# вывести доли клиентов с различным образованием
db.get_df("""
SELECT
    NAME_EDUCATION_TYPE,
    COUNT(*) AS client_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS share_percent
FROM application
GROUP BY NAME_EDUCATION_TYPE
ORDER BY share_percent DESC;
""")

Время выполнения: 0:00:00


,name_education_type,client_count,share_percent
0,Secondary / secondary special,252379,70.84
1,Higher education,87379,24.53
2,Incomplete higher,12001,3.37
3,Lower secondary,4291,1.20
4,Academic degree,205,0.06


In [ ]:
# подсчитать количество полных лет для клиентов, у которых есть во владении автомобиль и недвижимость.
# вывести топ 10 по возрастанию
db.get_df("""
SELECT
    FLOOR(ABS(DAYS_BIRTH) / 365.25) AS full_years
FROM application
WHERE FLAG_OWN_CAR = 'Y'
  AND FLAG_OWN_REALTY = 'Y'
ORDER BY full_years ASC
LIMIT 10;
""")

Время выполнения: 0:00:00


,full_years
0,20.0
1,20.0
2,21.0
3,21.0
4,21.0
5,21.0
6,21.0
7,21.0
8,21.0
9,21.0


In [ ]:
# вывести тех клиентов, у кого доход на одного члена семьи в два раза больше, чем в среднем на одного члена семьи по выборке
db.get_df("""
SELECT
    SK_ID_CURR,
    AMT_INCOME_TOTAL,
    CNT_FAM_MEMBERS,
    ROUND((AMT_INCOME_TOTAL / CNT_FAM_MEMBERS)::numeric, 2) AS income_per_family_member
FROM application
WHERE CNT_FAM_MEMBERS IS NOT NULL
  AND CNT_FAM_MEMBERS > 0
  AND (AMT_INCOME_TOTAL / CNT_FAM_MEMBERS) >
      2 * (
          SELECT AVG(AMT_INCOME_TOTAL / CNT_FAM_MEMBERS)
          FROM application
          WHERE CNT_FAM_MEMBERS IS NOT NULL
            AND CNT_FAM_MEMBERS > 0
      )
ORDER BY income_per_family_member DESC;
""")

Время выполнения: 0:00:00


,sk_id_curr,amt_income_total,cnt_fam_members,income_per_family_member
0,114967,117000000.0,3.0,39000000.0
1,385674,13500000.0,2.0,6750000.0
2,336147,18000090.0,4.0,4500020.0
3,337151,4500000.0,1.0,4500000.0
4,219563,4500000.0,1.0,4500000.0
...,...,...,...,...
27728,276208,375750.0,2.0,187875.0
27729,270540,751500.0,4.0,187875.0
27730,131255,187650.0,1.0,187650.0
27731,133844,187650.0,1.0,187650.0


Для каждого клиента был рассчитан доход на одного члена семьи как отношение общего дохода клиента (`AMT_INCOME_TOTAL`) к количеству членов семьи (`CNT_FAM_MEMBERS`).

Затем было вычислено среднее значение данного показателя по всей выборке.

Из таблицы были выбраны только те клиенты, у которых доход на одного члена семьи превышает среднее значение по выборке более чем в два раза.

In [16]:
# вывести клиентов старше 60 лет по которым нет данных в bureau
db.get_df("""
SELECT
    a.SK_ID_CURR,
    FLOOR(ABS(a.DAYS_BIRTH) / 365.25) AS age_years
FROM application a
LEFT JOIN bureau b
    ON a.SK_ID_CURR = b.SK_ID_CURR
WHERE FLOOR(ABS(a.DAYS_BIRTH) / 365.25) > 60
  AND b.SK_ID_CURR IS NULL
ORDER BY age_years DESC;
""")

Время выполнения: 0:00:00


,sk_id_curr,age_years
0,406901,69.0
1,270037,68.0
2,150827,68.0
3,291902,68.0
4,112756,68.0
...,...,...
5011,331624,61.0
5012,222738,61.0
5013,380448,61.0
5014,199858,61.0


In [ ]:
# вывести женщин, у которых в истории bureau было больше двух кредитов, просроченных на 61 день и более
# отсортировать в порядке убывания по кол-ву таких кредитов
db.get_df("""
SELECT
    a.SK_ID_CURR,
    COUNT(*) AS overdue_credit_count
FROM application a
JOIN bureau b
    ON a.SK_ID_CURR = b.SK_ID_CURR
WHERE a.CODE_GENDER = 'F'
  AND b.CREDIT_DAY_OVERDUE >= 61
GROUP BY a.SK_ID_CURR
HAVING COUNT(*) > 2
ORDER BY overdue_credit_count DESC;
""")

Время выполнения: 0:00:00


,sk_id_curr,overdue_credit_count
0,264144,5
1,142384,4
2,374345,3
3,375724,3
4,431820,3
5,114166,3
6,436084,3
7,337741,3


In [ ]:
# по данным из bureau (БКИ) рассчитать долю просрочки в активных займах
# вывести топ-7 мужчин с наибольшей суммой просрочки

db.get_df("""
WITH bureau_agg AS (
    SELECT
        SK_ID_CURR,

        -- сумма просрочки только по активным кредитам
        SUM(
            CASE
                WHEN CREDIT_ACTIVE = 'Active'
                    THEN COALESCE(AMT_CREDIT_SUM_OVERDUE, 0)
                ELSE 0
            END
        ) AS overdue_active_sum,

        -- сумма только активных кредитов
        SUM(
            CASE
                WHEN CREDIT_ACTIVE = 'Active'
                    THEN COALESCE(AMT_CREDIT_SUM, 0)
                ELSE 0
            END
        ) AS active_credit_sum,

        -- сумма всех кредитов клиента (активных и закрытых)
        SUM(COALESCE(AMT_CREDIT_SUM, 0)) AS total_credit_sum

    FROM bureau
    GROUP BY SK_ID_CURR
)

SELECT
    a.SK_ID_CURR,

    -- сумма просрочки по активным займам
    b.overdue_active_sum,

    -- сумма активных кредитов
    b.active_credit_sum,

    -- сумма всех кредитов клиента
    b.total_credit_sum,

    -- доля просрочки в активных кредитах
    ROUND(
        CASE
            WHEN b.active_credit_sum > 0
                THEN (b.overdue_active_sum / b.active_credit_sum)::numeric
            ELSE NULL
        END,
        4
    ) AS overdue_share_in_active_loans

FROM application a

-- присоединяем агрегаты по БКИ
JOIN bureau_agg b
    ON a.SK_ID_CURR = b.SK_ID_CURR

-- оставляем только мужчин
WHERE a.CODE_GENDER = 'M'

-- сортируем по максимальной сумме просрочки
ORDER BY b.overdue_active_sum DESC

-- выводим топ-7 клиентов
LIMIT 7;
""")

Время выполнения: 0:00:00


,sk_id_curr,overdue_active_sum,active_credit_sum,total_credit_sum,overdue_share_in_active_loans
0,435405,3681063.00,3690000.0,4054000.5,0.9976
1,427996,1571697.00,4231615.5,7431583.5,0.3714
2,394113,1332472.50,1530000.0,1825801.0,0.8709
3,266765,1224474.90,1350000.0,1421955.0,0.9070
4,167085,780192.00,158404.5,176404.5,4.9253
5,154595,742491.00,990000.0,4245754.5,0.7500
6,262411,709669.25,1318500.0,1773000.0,0.5382


### Фичи

In [ ]:
# признак 1 (отношение кредита к доходу)
db.get_df("""
SELECT
    SK_ID_CURR,
    ROUND((AMT_CREDIT / NULLIF(AMT_INCOME_TOTAL, 0))::numeric, 4) AS credit_to_income_ratio
FROM application;
""")

Время выполнения: 0:00:01


,sk_id_curr,credit_to_income_ratio
0,104120,4.0544
1,106607,7.4622
2,107395,2.8641
3,108120,3.6124
4,109741,0.9653
...,...,...
356250,456221,3.3956
356251,456222,3.9518
356252,456223,1.5556
356253,456224,2.0000


In [7]:
# признак 2 (отношение кредита к доходу)

db.get_df("""
SELECT
    SK_ID_CURR,
    ROUND((AMT_ANNUITY / NULLIF(AMT_INCOME_TOTAL, 0))::numeric, 4) AS annuity_to_income_ratio
FROM application;
""")

Время выполнения: 0:00:01


,sk_id_curr,annuity_to_income_ratio
0,104120,0.4183
1,106607,0.2824
2,107395,0.1270
3,108120,0.1852
4,109741,0.1023
...,...,...
356250,456221,0.1438
356251,456222,0.2026
356252,456223,0.1640
356253,456224,0.1117


In [ ]:
# признак 3 (количество активных кредитов)
db.get_df("""
SELECT
    SK_ID_CURR,
    COUNT(*) AS active_credit_count
FROM bureau
WHERE CREDIT_ACTIVE = 'Active'
GROUP BY SK_ID_CURR;
""")

Время выполнения: 0:00:00


,sk_id_curr,active_credit_count
0,233338,3
1,129976,3
2,157514,2
3,439741,2
4,214081,1
...,...,...
251810,342068,3
251811,158829,1
251812,417703,5
251813,279693,1


In [10]:
# признак 4  (длина кредитной истории)
db.get_df("""
SELECT
    SK_ID_CURR,
    ABS(MIN(DAYS_CREDIT)) AS credit_history_days
FROM bureau
GROUP BY SK_ID_CURR;
""")

Время выполнения: 0:00:00


,sk_id_curr,credit_history_days
0,233338,2414
1,129976,1105
2,174416,492
3,157514,2725
4,439741,738
...,...,...
305806,342068,1429
305807,158829,1516
305808,417703,869
305809,279693,1791


In [11]:
# признак 5 (Количество предыдущих заявок клиента)
db.get_df("""
SELECT
    SK_ID_CURR,
    COUNT(*) AS prev_application_count
FROM previous_application
GROUP BY SK_ID_CURR;

""")

Время выполнения: 0:00:01


,sk_id_curr,prev_application_count
0,146688,1
1,371316,6
2,233338,5
3,337824,2
4,159965,3
...,...,...
338852,312236,2
338853,158829,10
338854,155703,10
338855,417703,16
